In [ ]:
!pip -q install google-search-results pandas

import pandas as pd
from serpapi import GoogleSearch
from getpass import getpass
from urllib.parse import urlparse
import json
import time

SERPAPI_API_KEY = getpass("Enter your SerpApi API key: ")

In [ ]:
BAD_DOMAINS = {
    "tripadvisor.com",
    "foodora.hu",
    "wolt.com",
    "ubereats.com",
    "facebook.com",
    "instagram.com",
    "tiktok.com",
    "youtube.com",
    "foursquare.com",
    "yelp.com",
    "restaurantguru.com",
    "welovebudapest.com"
}

def clean_domain(url):
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return None

def is_likely_official(url):
    d = clean_domain(url)
    return bool(d) and d not in BAD_DOMAINS

def extract_website_from_item(item):
    if not isinstance(item, dict):
        return None

    links = item.get("links")
    if isinstance(links, dict):
        website = links.get("website")
        if isinstance(website, str) and website.startswith("http"):
            return website

    for key in ["website", "website_link", "link"]:
        val = item.get(key)
        if isinstance(val, str) and val.startswith("http"):
            return val

    return None

In [ ]:
def find_official_website(title, address):
    query = f'{title} {address} official website'
    params = {
        "engine": "google",
        "q": query,
        "api_key": SERPAPI_API_KEY,
        "num": 10
    }

    try:
        search = GoogleSearch(params)
        results = search.get_dict()
        organic = results.get("organic_results", [])

        for item in organic:
            link = item.get("link")
            if link and is_likely_official(link):
                return link
    except Exception:
        pass

    return None

In [ ]:
def search_restaurants_with_websites(query="restaurants", location="Budapest, Hungary",
                                     target_count=10, max_pages=3):
    collected = []
    seen_titles = set()

    start = 0
    page = 0

    while len(collected) < target_count and page < max_pages:
        params = {
            "engine": "google_local",
            "q": query,
            "location": location,
            "api_key": SERPAPI_API_KEY,
            "start": start
        }

        search = GoogleSearch(params)
        results = search.get_dict()

        local_results = results.get("local_results", [])
        if isinstance(local_results, dict):
            items = local_results.get("places", []) or local_results.get("place", [])
        elif isinstance(local_results, list):
            items = local_results
        else:
            items = []

        if not items:
            break

        for item in items:
            title = item.get("title")
            address = item.get("address")

            unique_key = f"{title} | {address}"
            if unique_key in seen_titles:
                continue
            seen_titles.add(unique_key)

            website = extract_website_from_item(item)

            # Fallback via Google Search API
            if not website:
                website = find_official_website(title or "", address or "")

            if not website:
                print(f"Skipping (no website found): {title}")
                continue

            gps = item.get("gps_coordinates", {}) if isinstance(item, dict) else {}

            row = {
                "title": title,
                "type": item.get("type"),
                "address": address,
                "phone": item.get("phone"),
                "description": item.get("description"),
                "rating": item.get("rating"),
                "reviews": item.get("reviews"),
                "price": item.get("price"),
                "hours": json.dumps(item.get("hours"), ensure_ascii=False) if item.get("hours") else None,
                "website": website,
                "website_domain": clean_domain(website),
                "directions": item.get("links", {}).get("directions") if isinstance(item.get("links"), dict) else None,
                "place_id": item.get("place_id"),
                "latitude": gps.get("latitude"),
                "longitude": gps.get("longitude"),
                "source_has_direct_website": bool(extract_website_from_item(item)),
                "used_google_search_website": bool(not extract_website_from_item(item) and website)
            }

            print(f"Keeping: {title} -> {website}")
            collected.append(row)

            if len(collected) >= target_count:
                break

            time.sleep(0.2)

        page += 1
        start += 20

    return pd.DataFrame(collected)

In [ ]:
location = input("Enter location (example: Budapest, Hungary): ").strip() or "Budapest, Hungary"
query = input("Enter query (example: restaurants, pizza, cafes): ").strip() or "restaurants"
target_count = int(input("How many rows WITH websites do you want? (example 10): ").strip() or "10")

df = search_restaurants_with_websites(
    query=query,
    location=location,
    target_count=target_count,
    max_pages=3
)

print(f"Collected {len(df)} rows with websites")
display(df)

df.to_csv("nearby_restaurants_with_websites.csv", index=False)
df.to_json("nearby_restaurants_with_websites.json", orient="records", force_ascii=False, indent=2)

print("Saved nearby_restaurants_with_websites.csv")
print("Saved nearby_restaurants_with_websites.json")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================================================
# 1) Install packages
# =========================================================
!pip -q install pandas requests beautifulsoup4 lxml

# =========================================================
# 2) Imports
# =========================================================
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import time

# =========================================================
# 3) Config
# =========================================================
INPUT_FILE = "/content/nearby_restaurants_with_websites.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

SCAN_PATHS = [
    "",
    "/menu",
    "/menus",
    "/food-menu",
    "/drinks",
    "/offers",
    "/offer",
    "/deals",
    "/deal",
    "/promotions",
    "/promotion",
    "/discount",
    "/discounts",
    "/specials",
    "/happy-hour"
]

MENU_KEYWORDS = [
    "menu", "food menu", "drink menu", "lunch menu", "dinner menu",
    "our menu", "view menu"
]

OFFER_KEYWORDS = [
    "offer", "offers", "deal", "deals", "promotion", "promo",
    "special", "specials", "happy hour", "limited time", "combo"
]

DISCOUNT_KEYWORDS = [
    "discount", "% off", "save", "coupon", "voucher", "student", "student id"
]